In [4]:
# TP1 : 3 Premier notebook pandas : chargement (Parquet), dimensions, types, mémoire, taux de remplissage par colonne.

# PyArrow est la bibliothèque idéale car elle accède directement aux métadonnées 
# sans charger toutes les données en mémoire (Pour éviter tout Crash de RAM).
import pyarrow.parquet as pq

parquet_file_path = "../../food.parquet"
# Chargement du Parquet : Le parcourir en mode streaming (Une lecture en flux continu) :
parquet_file = pq.ParquetFile(parquet_file_path)

# Les métadonnées d'un fichier Parquet sont un bloc d'informations situé à la fin du fichier (le footer) qui décrit toute la structure, l'organisation et le contenu des données stockées, sans qu'il soit nécessaire de lire les données elles-mêmes.
metadata_parquet_file = parquet_file.metadata

# Les dimensions du parquet :
number_of_rows = metadata_parquet_file.num_rows
number_of_columns = metadata_parquet_file.num_columns
print(f"Ce fichier comprend {number_of_rows} lignes, et {number_of_columns} colonnes.\n")

# Les types : (Avec plus de détails)
schema = parquet_file.schema
print(f"Les types de données avec tout le détail :\n{schema}")

# Taille compressée totale (sur le disque, selon les métadonnées)
compressed_size = metadata_parquet_file.serialized_size

uncompressed_bytes = sum(
    metadata_parquet_file.row_group(i).total_byte_size
    for i in range(metadata_parquet_file.num_row_groups)
)
print(f"Taille estimée en mémoire (décompressée) : {uncompressed_bytes / (1024 * 1024):.2f} Mo")


Ce fichier comprend 4636471 lignes, et 145 colonnes.

Les types de données avec tout le détail :
required group field_id=-1 schema {
  optional int32 field_id=-1 additives_n;
  optional group field_id=-1 additives_tags (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional group field_id=-1 allergens_tags (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional group field_id=-1 brands_tags (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional binary field_id=-1 brands (String);
  optional binary field_id=-1 categories (String);
  optional group field_id=-1 categories_tags (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional group field_id=-1 categories_properties {
    optional int32 field_id=-1 ciqual_food_code;
    optional in

In [5]:
import pyarrow as pa
import numpy as np
import pandas as pd

# Définir la taille maximale de chaque lot de lignes à charger en mémoire
TAILLE_BLOC = 10000

products_sold_infr = 0
filled_nutri_score = 0
filled_nutri_score_fr = 0
filled_nutri_grade = 0

result_df_filtered_ffr = pd.DataFrame()
blocs_list = []

# Itérer séquentiellement sur le fichier par blocs de lignes
for batch in parquet_file.iter_batches(batch_size=TAILLE_BLOC, columns=['countries_tags', 'nutriscore_grade', 'nutriscore_score', 'brands', 'nutriments']):
    # Convertir le batch/bloc en Pandas pour modifier
    data_frame = batch.to_pandas()

    # 4.1 Combien de produits vendus en France ? => products_sold_infr
    filtered_data = data_frame[(data_frame["countries_tags"].notna()) & data_frame["countries_tags"].str.contains('en:france', regex=False)]
    products_sold_infr += len(filtered_data)

    # 4.2 Quelle part a un Nutri-Score renseigné ? => Que la France
    filtered_data_fr = filtered_data[filtered_data["nutriscore_score"].notna()]
    filled_nutri_score_fr += len(filtered_data_fr)
    
    result_df_filtered_ffr = pd.concat([result_df_filtered_ffr, filtered_data_fr], ignore_index=True)

    # 4.2 Quelle part a un Nutri-Score renseigné ? => Tout le parquet,  en se basant sur nutriscore_score
    filtered_data = data_frame[data_frame["nutriscore_score"].notna()]
    filled_nutri_score += len(filtered_data)

    # 4.2 Quelle part a un Nutri-Score renseigné ? => Tout le parquet, en se basant sur nutriscore_grade
    # filtered_data = data_frame[(data_frame["nutriscore_grade"].notna()) & (data_frame["nutriscore_grade"].str.strip() != "") & (~data_frame["nutriscore_grade"].isin(['unknown', 'not-applicable']))]
    # filled_nutri_grade += len(filtered_data)

# print(products_sold_infr)
# print(filled_nutri_score)
# print(filled_nutri_score_fr)
# print(filled_nutri_grade)
print(len(result_df_filtered_ffr))

# 2-La proportion des nutriscores rensignés :
# 2.1-La proportion des nutriscores rensignés 'nutriscore_grade', sans les 'None', sans les 'unknown' et 'not-applicable' (Pour tout le parquet) :
# proportion_grade_filled = (filled_nutri_grade / number_of_rows) * 100
# print(filled_nutri_grade) # 1381069
# print(f"{proportion_grade_filled:.2f}%") # 29.79%

# 2.2-La proportion des nutriscores rensignés 'nutriscore_score' (Pour tout le parquet) :
proportion_score_not_none = (filled_nutri_score / number_of_rows) * 100
print(filled_nutri_score) # 1381069
print(f"{proportion_score_not_none:.2f}%") # 29.79%

# 2.3-La proportion des nutriscores rensignés 'nutriscore_score' sans les 'None' (Pour les produits vendus en France seulement) :
proportion_score_not_none_fr = (filled_nutri_score_fr / products_sold_infr) * 100
print(filled_nutri_score_fr) # 463757
print(f"{proportion_score_not_none_fr:.2f}%") # 37.18%

print("########################## Ma Data_Frame aprés le filtre c'est bien 'result_df_filtered_ffr' ##########################")

463757
1381069
29.79%
463757
37.18%
########################## Ma Data_Frame aprés le filtre c'est bien 'result_df_filtered_ffr' ##########################


In [11]:
from tabulate import tabulate

donnees = [
    [products_sold_infr, filled_nutri_score_fr, "{:.2f}%".format(proportion_score_not_none_fr), filled_nutri_score, "{:.2f}%".format(proportion_score_not_none)]
]

headers = ["Produits vendus en France", "Total des nutriscores renseignés/France", "Proportion des nutriscores rensignés/France", "Total des nutriscores renseignés/Parquet", "Proportion des nutriscores rensignés/Parquet"]

 # --- AFFICHAGE DU TABLEAU DES DONNEES ---
# Affichage du tableau avec des bordures en grille
print(tabulate(donnees, headers=headers, tablefmt="grid"))

+-----------------------------+-------------------------------------------+-----------------------------------------------+--------------------------------------------+------------------------------------------------+
|   Produits vendus en France |   Total des nutriscores renseignés/France | Proportion des nutriscores rensignés/France   |   Total des nutriscores renseignés/Parquet | Proportion des nutriscores rensignés/Parquet   |
+=============================+===========================================+===============================================+============================================+================================================+
|                     1247336 |                                    463757 | 37.18%                                        |                                    1381069 | 29.79%                                         |
+-----------------------------+-------------------------------------------+-----------------------------------------------+-----

In [12]:
from collections import Counter

# 4.3-Les dix marques les plus présentes :
count_brands = Counter()

# Nettoyage : suppression des valeurs manquantes ou vides
result_df_filtered_ffr = result_df_filtered_ffr[(result_df_filtered_ffr["brands"].str.strip() != "") & (result_df_filtered_ffr["brands"].notna())]

brands_serie = result_df_filtered_ffr["brands"].str.split(',').explode().str.strip()
count_brands.update(brands_serie)

# 4.3-Extraction et affichage du Top 10 :
brands_10 = count_brands.most_common(10)

print("Les 10 marques les plus présentes avec un Nutri-Score en France :")
for brand, total in brands_10:
    print(f"- {brand} : {total} produits")

Les 10 marques les plus présentes avec un Nutri-Score en France :
- U : 10211 produits
- Carrefour : 9529 produits
- Auchan : 6195 produits
- Marque Repère : 5389 produits
- Casino : 4963 produits
- Leader Price : 4179 produits
- Monoprix : 3267 produits
- Lidl : 3246 produits
- Nestlé : 3241 produits
- Picard : 3111 produits


In [ ]:
import re

# Cette regex cherche le texte 'name': 'sugars' et extrait la valeur numérique après '100g':
regex_sucre = re.compile(r"'name':\s*'sugars'.*?'100g':\s*([0-9.]+)")

# Cette regex cherche le texte 'name': 'salt' et extrait la valeur numérique après '100g':
regex_salt = re.compile(r"'name':\s*'salt'.*?'100g':\s*([0-9.]+)")

# Cette regex cherche le texte 'name': 'salt' et extrait la valeur numérique après '100g':
regex_energy = re.compile(r"'name':\s*'energy'.*?'100g':\s*([0-9.]+)")

def extract_value_or_nan(column, regex):
    try:
        if pd.isna(column).any():
            return None
    except Exception:
        if pd.isna(column):
            return None
    column_str = str(column)
    match = regex.search(column_str)
    if match:
        return float(match.group(1))
    
    return None  # Retourne None si la valeur de 'regex' n'est pas trouvé dans la cellule

# Création d'une Series temporaire ultra-légère
sugar_series = pd.Series([extract_value_or_nan(val, regex_sucre) for val in result_df_filtered_ffr['nutriments']])
salt_series = pd.Series([extract_value_or_nan(val,regex_salt) for val in result_df_filtered_ffr['nutriments']])
energy_series = pd.Series([extract_value_or_nan(val, regex_energy) for val in result_df_filtered_ffr['nutriments']])

# 4.4-Calcul des taux des manquants sur les nutriments clés (sugars_100g, salt_100g, energy_100g)
rate_sugar_missed = sugar_series.isna().mean() * 100
rate_salt_missed = salt_series.isna().mean() * 100
rate_energy_missed = energy_series.isna().mean() * 100

print(f"Taux de valeurs manquantes pour 'sugars' : {rate_sugar_missed:.2f}%")
print(f"Taux de valeurs manquantes pour 'salt' : {rate_salt_missed:.2f}%")
print(f"Taux de valeurs manquantes pour 'energy' : {rate_energy_missed:.2f}%")


Taux de valeurs manquantes pour 'sugars' : 0.73%
Taux de valeurs manquantes pour 'salt' : 0.66%
Taux de valeurs manquantes pour 'energy' : 0.67%
